# X1 · Git, DVC and MLflow: reproducible experiments

| Session | Time | Previous | Next |
| --- | --- | --- | --- |
| Self-study, not part of the live sessions | about 60 minutes | [12 · Capstone](12_capstone.ipynb) | [X2 · Building a scikit-learn compatible estimator](X2_sklearn_compatible_estimator.ipynb) |

Three months after a successful model went live, somebody asks: *"Which data was the model trained on, with
which code and which settings, and how good was it?"* If you cannot answer in five minutes, the model is not
reproducible. This notebook shows the three tools data teams commonly combine to answer that question:

- **Git** versions the code (and small text files such as configs),
- **DVC** versions data and models, which are too large for Git, and can re-run pipelines,
- **MLflow** records every experiment: parameters, metrics and artefacts.

> **Learning goals.** By the end you can
>
> - decide which files of a data science project belong in Git and which do not
> - write commit messages that explain what changed and why the results changed
> - explain what DVC adds to Git and read a `dvc.yaml` pipeline
> - log a training run of the churn model with MLflow and compare runs
> - design a reproducible workflow combining the three tools

**Before you start.** Notebooks 07 (config dataclasses) and 11 (the package). DVC and MLflow are **optional**
dependencies because they are large. Install them only if you want to run the MLflow cells:

```bash
uv sync --extra tracking
```

Without them, the notebook still runs: the MLflow cells print what they would do.

**In this notebook**

- Theory 1 · Git habits for data science
- Hands-on A · What belongs in Git?
- Theory 2 · DVC: version control for data and models
- Theory 3 · MLflow: tracking experiments
- Theory 4 · How the pieces fit together
- Hands-on B · A reproducible churn workflow

---

## Theory 1 · Git habits for data science

You should know the basics (`git add`, `commit`, `push`, `pull`, branches). If not, work through
https://learngitbranching.js.org/ first. Here we focus on what is specific to data science work.

### Commit messages that record results

Git records *what* changed. Only you can record *why*, and in data science "why" usually includes the effect on
the model. Compare:

```text
update model
```

```text
Use contract type and payment method as features

Holdout ROC AUC 0.834 -> 0.840 (configs/baseline.json, same split).
Month-to-month customers churn 15 times more often than two-year customers.
Tenure already captures much of that, hence the small gain.
```

The second message lets a colleague (or you, in six months) understand a change without rerunning anything.

### Branches

Keep the main branch working at all times. Do each piece of work (a new feature, a bug fix, an experiment) on
its own branch and merge it when it is reviewed and the tests pass. Experiments that did not work out can live on
their branch: you keep the history without cluttering main.

### What not to commit

Git is built for text files that change line by line. It handles large binary files badly, and anything you
commit stays in the history forever, even after you delete it. Never commit:

- **large data** (CSV, Parquet) and **model files** (`.pkl`, `.joblib`, `.pt`): use DVC or a data store,
- **secrets** (passwords, API keys, `.env` files): use environment variables or a secret manager,
- **generated files** (`__pycache__`, `.ipynb_checkpoints`, `mlruns/`): they can be recreated.

Add the `.gitignore` **before** your first commit. Here is the one from this repository:

In [ ]:
from pathlib import Path

REPO = Path("..").resolve()
print((REPO / ".gitignore").read_text())

Note one deliberate exception: this workshop commits `data/raw/telco_churn.csv` (about 1 MB, open licence), so
the course works offline. In a real project that file would be tracked with DVC.

### Notebooks in Git

Notebook files contain outputs (tables, images) that change on every run and make diffs unreadable. Two common
habits: clear outputs before committing (this repository's notebooks are committed without outputs), and keep
logic in the package so notebook changes stay small.

---

## Hands-on A · What belongs in Git?

A real project mixes code, configuration, data, models, secrets and generated files. Deciding which of them go
into Git is a habit you apply on every commit, so practise it on a typical churn project.

### Exercise A.1

*About 10 minutes, including the discussion.*

For each path below, decide whether it should be committed to Git. Put the paths that **should** be committed
into the set `commit`.

**What your code should do.**

- `commit` is a Python set of strings, e.g. `{"src/churn/model.py", "pyproject.toml"}`.
- It contains only paths from the list `paths`, and exactly those that belong in Git; everything else stays out.
- You can type the set by hand, or build it from `paths` with a set comprehension:
  `{p for p in paths if <your rule>}`.

**Discuss.** This workshop commits `data/raw/telco_churn.csv`, yet a data export should not be committed. Where exactly would you draw the line in your own projects (size, licence, personal data, how often it changes)? And what would you do if a password had already been pushed?

In [ ]:
# Exercise A.1: your code here

paths = [
    "src/churn/model.py",
    "configs/baseline.json",
    "data/raw/customers_2026_10.csv",  # 4 GB export from the data warehouse
    "models/churn_rf.joblib",
    ".env",  # contains the database password
    "tests/test_model.py",
    "pyproject.toml",
    "uv.lock",
    "notebooks/01_eda.ipynb",
    "mlruns/0/meta.yaml",
    "data/raw/customers_2026_10.csv.dvc",  # small pointer file written by DVC
]

commit = set()  # Your code here

In [ ]:
# Check for exercise A.1: run after solving it
expected = {
    "src/churn/model.py",
    "configs/baseline.json",
    "tests/test_model.py",
    "pyproject.toml",
    "uv.lock",
    "notebooks/01_eda.ipynb",
    "data/raw/customers_2026_10.csv.dvc",
}
missing, extra = expected - commit, commit - expected
assert not extra, f"These should NOT be committed: {sorted(extra)}"
assert not missing, f"These SHOULD be committed: {sorted(missing)}"
print("All checks passed")

<details><summary><b>Show solution and reasons</b></summary>

Commit: code, tests, configs, `pyproject.toml`, `uv.lock` (exact versions make environments reproducible), the
notebook (ideally with outputs cleared) and the small `.dvc` pointer file.

Do not commit: the 4 GB export (DVC tracks it; the pointer file goes into Git), the model file (an artefact,
tracked with DVC or MLflow), `.env` (a secret: once pushed, consider the password leaked and change it), and
`mlruns/` (MLflow's local store, recreated by MLflow).

</details>

---

## Theory 2 · DVC: version control for data and models

Git cannot answer "which version of the data was this model trained on?" because the data is not in Git. DVC
(Data Version Control) closes that gap:

- `dvc add data/raw/customers.csv` computes a hash of the file, moves the file into a cache and writes a tiny
  pointer file `customers.csv.dvc` containing that hash. You commit the pointer file to Git.
- `dvc push` uploads the data to a **remote** (S3, Google Cloud Storage, Azure, an SSH server or a shared
  folder). `dvc pull` downloads exactly the version that the current Git commit points to.
- Checking out an old commit and running `dvc checkout` restores the matching data.

The basic workflow (run in a terminal, not in this notebook):

```bash
uv sync --extra tracking            # installs dvc into the project environment
uv run dvc init                     # once per repository, creates .dvc/
uv run dvc add data/raw/customers.csv
git add data/raw/customers.csv.dvc data/raw/.gitignore
git commit -m "Track October customer export with DVC"

uv run dvc remote add -d storage s3://my-team-bucket/dvc-store
uv run dvc push                     # upload data to the remote

# a colleague, on another machine
git clone <repo-url> && cd <repo>
uv run dvc pull                     # download the data version of this commit
```

### DVC pipelines

DVC can also describe how outputs are produced from inputs. Each **stage** lists its command, the files it
depends on and the files it produces. `dvc repro` then re-runs only the stages whose dependencies changed, and
records the exact hashes in `dvc.lock` (committed to Git).

```yaml
# dvc.yaml
stages:
  train:
    cmd: python -m churn --config configs/baseline.json --save models/churn.joblib
    deps:
      - data/raw/telco_churn.csv
      - configs/baseline.json
      - src/churn
    outs:
      - models/churn.joblib
```

```bash
uv run dvc repro      # runs "train" only if the data, the config or the code changed
uv run dvc dag        # shows the pipeline as a graph
```

Notice how well this fits the package from notebook 11: because the project has a command line entry point,
the pipeline stage is a single, readable command.

---

## Theory 3 · MLflow: tracking experiments

DVC answers "which data and code produced this model?". MLflow answers "which of my 40 runs was best, and with
which settings?". For every run it stores:

| Concept | What it stores | Churn example |
|---|---|---|
| Parameters | settings, logged once per run | `model=random_forest`, `threshold=0.3` |
| Metrics | numbers, optionally per step | `roc_auc=0.84`, `recall=0.78` |
| Artefacts | files | the saved `ChurnModel`, a confusion matrix plot |
| Tags | free-form labels | `dataset=telco_2026_10`, `author=...` |

---

## Theory 4 · How the pieces fit together

```mermaid
flowchart LR
    code["Code and configs<br/>(src/, configs/)"] -->|git commit| git[(Git)]
    data["Data and models<br/>(data/, models/)"] -->|dvc add / dvc push| remote[(DVC remote)]
    data -. pointer files .-> git
    run["Training run<br/>python -m churn"] -->|params, metrics, artefacts| mlflow[(MLflow)]
    git --> run
    remote --> run
```

A typical cycle:

```bash
git switch -c contract-features            # 1. a branch for the idea
# ... edit src/churn or configs ...
uv run pytest                              # 2. tests still pass
uv run dvc repro                           # 3. retrain what changed (and log to MLflow)
uv run mlflow ui                           # 4. compare with earlier runs
git add src configs dvc.lock metrics
git commit -m "Add contract features: ROC AUC 0.834 -> 0.840"
git push && uv run dvc push                # 5. share code and data
```

Anyone can now check out that commit, run `dvc pull` and `dvc repro`, and get the same model.

---

## Hands-on B · A reproducible churn workflow

Now put the tools to work on the churn project. First a guided step: train the model with the `churn`
package and log the run.

The cell below uses a temporary folder as
the tracking store so that it leaves nothing behind. If MLflow is not installed, it only shows what would be
logged.

In [ ]:
import tempfile

from churn import CSVCustomerLoader, ExperimentConfig, run_experiment

DATA_PATH = REPO / "data" / "raw" / "telco_churn.csv"
loader = CSVCustomerLoader(DATA_PATH)


def config_as_params(config: ExperimentConfig) -> dict[str, str]:
    """Flatten a config into MLflow parameters (MLflow stores parameters as strings)."""
    params = {
        "model": config.model,
        "threshold": str(config.threshold),
        "test_size": str(config.test_size),
        "random_state": str(config.random_state),
        "features": ",".join(config.features),
    }
    params.update({f"model__{key}": str(value) for key, value in config.model_params.items()})
    return params


def result_as_metrics(result) -> dict[str, float]:
    cm = result.confusion
    return {
        "roc_auc": result.roc_auc,
        "precision": cm.precision,
        "recall": cm.recall,
        "f1": cm.f1,
        "savings": result.savings,
    }


config = ExperimentConfig(model="random_forest", model_params={"n_estimators": 200}, threshold=0.3)
model, result = run_experiment(config, loader=loader)

try:
    import mlflow
except ImportError:
    mlflow = None

if mlflow is None:
    print("MLflow is not installed (uv sync --extra tracking). This run would log:")
    print("  parameters:", config_as_params(config))
    print("  metrics:   ", {k: round(v, 3) for k, v in result_as_metrics(result).items()})
    print("  artefact:   the saved ChurnModel (churn.joblib)")
else:
    with tempfile.TemporaryDirectory() as tracking_dir:
        mlflow.set_tracking_uri(Path(tracking_dir).as_uri())
        mlflow.set_experiment("churn")
        with mlflow.start_run(run_name="random_forest_t0.3") as run:
            mlflow.log_params(config_as_params(config))
            mlflow.log_metrics(result_as_metrics(result))
            mlflow.set_tag("dataset", DATA_PATH.name)
            model_path = model.save(Path(tracking_dir) / "churn.joblib")
            mlflow.log_artifact(str(model_path))
        print(f"Logged run {run.info.run_id}: {result.summary()}")

Two design choices worth noticing:

- The training code (`run_experiment`) knows nothing about MLflow. Tracking is added *around* it, in the
  application layer. You can switch to another tracker (Weights & Biases, a plain JSON file) without touching
  the model code (notebook 08).
- Parameters come straight from the `ExperimentConfig` and metrics from the `ExperimentResult`. Because both are
  dataclasses with clear fields, logging them is a few lines. Untyped dictionaries scattered across a notebook
  make this much harder.

In a real project you would point `set_tracking_uri` at a shared MLflow server (or leave the default, a local
`mlruns/` folder that is ignored by Git) and open the comparison table with `uv run mlflow ui`.

### Exercise B.1
> If you finish early, try this one.

Compare thresholds 0.2, 0.3, 0.4 and 0.5 for the logistic regression model. Write a function
`threshold_sweep(thresholds)` that runs one experiment per threshold and returns a list of dicts with the
threshold and the metrics from `result_as_metrics`. If MLflow is installed, also log each run (one run per
threshold, inside a temporary tracking directory). Which threshold saves the most money?

**What your code should do.**

- Input: a list of floats such as `[0.2, 0.3, 0.4, 0.5]`.
- Output: a list with one dict per threshold, in the same order, e.g.
  `{"threshold": 0.3, "roc_auc": 0.84, "precision": 0.52, "recall": 0.78, "f1": 0.62, "savings": 24400.0}`.
- Each run uses `ExperimentConfig(threshold=t)` and the same `loader`, so all runs share the same split.
- Without MLflow installed the function still works; it only skips the logging.
- Useful Python: `{"threshold": t, **metrics}` builds a new dict with the key `"threshold"` plus every entry of
  the dict `metrics`.

**Discuss.** The best threshold depends on the offer cost and acceptance rate in `RetentionCost`. Who in a company should own those numbers, and how would you record which assumptions a logged run used?

In [ ]:
# Exercise B.1: your code here


def threshold_sweep(thresholds: list[float]) -> list[dict[str, float]]:
    raise NotImplementedError("Your code here")

In [ ]:
# Check for exercise B.1: run after solving it
import pandas as pd

rows = threshold_sweep([0.2, 0.3, 0.4, 0.5])
table = pd.DataFrame(rows)
assert len(table) == 4
assert {"threshold", "recall", "savings"} <= set(table.columns)
assert table.sort_values("threshold")["recall"].is_monotonic_decreasing, (
    "a higher threshold should never flag more churners"
)
print(table.round(3))
print("All checks passed")

<details><summary><b>Show hint</b></summary>

Create a new `ExperimentConfig(threshold=t)` for each threshold and reuse the same `loader`. The split is the
same in every run because `random_state` and `test_size` do not change, so the comparison is fair.

</details>

<details><summary><b>Show solution</b></summary>

```python
def threshold_sweep(thresholds: list[float]) -> list[dict[str, float]]:
    rows = []
    for t in thresholds:
        config = ExperimentConfig(threshold=t)
        _, result = run_experiment(config, loader=loader)
        rows.append({"threshold": t, **result_as_metrics(result)})
        if mlflow is not None:
            with mlflow.start_run(run_name=f"logreg_t{t}"):
                mlflow.log_params(config_as_params(config))
                mlflow.log_metrics(result_as_metrics(result))
    return rows
```

With MLflow, wrap the loop in `with tempfile.TemporaryDirectory() as d: mlflow.set_tracking_uri(Path(d).as_uri())`
as in the cell above. With the default cost assumptions a low threshold (around 0.2 to 0.3) saves the most: an
offer is cheap compared with a lost customer, so flagging too many is less costly than missing churners. Choosing
the threshold on the test set and reporting the test result is optimistic; in practice choose it on a
validation split.

</details>

### Exercise B.2
> **Challenge**

Design a reproducible pipeline for this repository, and make it executable as data so you can test your reasoning.

1. Write the `dvc.yaml` you would use as a Python dict `PIPELINE` with the same structure as the YAML above: two
   stages, `train` (as above, but also writing metrics to `metrics/holdout.json`) and `evaluate_cost` (reads the
   saved model and the data, writes `metrics/cost.json`). Each stage has the keys `cmd`, `deps`, `outs` and
   `metrics` (lists of paths; a folder such as `src/churn` covers every file inside it).
2. Write `stages_to_rerun(changed_path, pipeline)` that returns, in pipeline order, the names of the stages
   `dvc repro` would re-run after `changed_path` changed: a stage re-runs if one of its `deps` is that path or a
   folder containing it, or if one of its `deps` is an output (`outs` or `metrics`) of a stage that re-runs.

**What your code should do.**

- `PIPELINE` is a nested dict: `{"stages": {"train": {...}, "evaluate_cost": {...}}}`, each stage with the keys
  `cmd` (a string) and `deps`, `outs`, `metrics` (lists of path strings), stages in execution order.
- `stages_to_rerun(changed_path, pipeline)` returns a list of stage names in pipeline order, e.g.
  `stages_to_rerun("configs/baseline.json", PIPELINE)` gives `["train", "evaluate_cost"]` and
  `stages_to_rerun("README.md", PIPELINE)` gives `[]`.
- A dependency that is a folder covers every file inside it: `"src/churn"` covers `"src/churn/model.py"`.
- Useful Python: `path.startswith(folder + "/")` tests whether a path lies inside a folder; looping over
  `pipeline["stages"].items()` visits the stages in the order they were written.

**Discuss.** What has to change in `src/churn/__main__.py` so that `train` can write `metrics/holdout.json`, and
which layer does that change belong to? Where would MLflow logging go in this setup, and what does it add that
`dvc metrics show` does not?

In [ ]:
# Exercise B.2: your code here

PIPELINE = {
    "stages": {
        "train": {"cmd": "...", "deps": [], "outs": [], "metrics": []},
        "evaluate_cost": {"cmd": "...", "deps": [], "outs": [], "metrics": []},
    }
}


def stages_to_rerun(changed_path, pipeline):
    raise NotImplementedError("Your code here")

In [ ]:
# Check for exercise B.2: run after solving it
train, cost = PIPELINE["stages"]["train"], PIPELINE["stages"]["evaluate_cost"]
assert {"data/raw/telco_churn.csv", "configs/baseline.json", "src/churn"} <= set(train["deps"])
assert "models/churn.joblib" in train["outs"] and "metrics/holdout.json" in train["metrics"]
assert "models/churn.joblib" in cost["deps"] and "metrics/cost.json" in cost["metrics"]

assert stages_to_rerun("configs/baseline.json", PIPELINE) == ["train", "evaluate_cost"]
assert stages_to_rerun("src/churn/model.py", PIPELINE) == ["train", "evaluate_cost"]
assert stages_to_rerun("README.md", PIPELINE) == []
print("All checks passed")

<details><summary><b>Show solution and discussion points</b></summary>

```python
PIPELINE = {
    "stages": {
        "train": {
            "cmd": "python -m churn --config configs/baseline.json --save models/churn.joblib"
            " --metrics-out metrics/holdout.json",
            "deps": ["data/raw/telco_churn.csv", "configs/baseline.json", "src/churn"],
            "outs": ["models/churn.joblib"],
            "metrics": ["metrics/holdout.json"],
        },
        "evaluate_cost": {
            "cmd": "python scripts/evaluate_cost.py",
            "deps": ["models/churn.joblib", "data/raw/telco_churn.csv", "src/churn/evaluation.py"],
            "outs": [],
            "metrics": ["metrics/cost.json"],
        },
    }
}


def stages_to_rerun(changed_path, pipeline):
    def covers(dep, path):
        return path == dep or path.startswith(dep.rstrip("/") + "/")

    changed = {changed_path}
    rerun = []
    for name, stage in pipeline["stages"].items():  # dicts keep insertion order
        if any(covers(dep, path) for dep in stage["deps"] for path in changed):
            rerun.append(name)
            changed |= set(stage["outs"]) | set(stage["metrics"])
    return rerun
```

Discussion points:

1. Metrics files are declared with `cache: false` in a real `dvc.yaml`, so they are stored in Git and diffable.
2. A new CLI option such as `--metrics-out` that writes the fields of the `ExperimentResult`. It is an
   application-layer change: `run_experiment` and the model stay untouched.
3. A change to `configs/baseline.json` re-runs both stages (the config is a dependency of `train`, and
   `evaluate_cost` depends on its output). The commit contains the config change, the updated `dvc.lock` and the
   metrics files; the model itself goes to the DVC remote with `dvc push`.
4. MLflow logging goes in the CLI (application layer), around `run_experiment`. It adds a searchable history of
   *all* runs, including exploratory ones that never become a commit, plus a UI to compare them.
   `dvc metrics show` compares the metrics of committed pipeline states.

</details>

---

## Summary

- Git versions code and configs; write commit messages that say what changed and what it did to the results.
- Never commit large data, model files, secrets or generated files; add `.gitignore` first.
- DVC keeps data and models out of Git but tied to Git commits through small pointer files, and re-runs only the
  pipeline stages whose inputs changed.
- MLflow records parameters, metrics and artefacts of every run so you can compare and reproduce them.
- Keep tracking code in the application layer; typed config and result objects make logging trivial.

**Common pitfalls**

- Committing a data file "just once": it stays in the history and bloats every clone.
- Logging metrics but not the data version or the config, which makes the run impossible to reproduce.
- Choosing hyperparameters or thresholds on the test set and reporting that score as the expected performance.

---

## Sources and further reading

**Sources.** This notebook is based on:

- [Scott Chacon and Ben Straub, *Pro Git*](https://git-scm.com/book/en/v2), the free reference book on Git.
- [DVC: Get started](https://dvc.org/doc/start) and [MLflow Tracking](https://mlflow.org/docs/latest/tracking.html), the official documentation.
- [github/gitignore](https://github.com/github/gitignore), templates for `.gitignore` files.

**For self-study.** Where to go next if you want to learn more:

- [Learn Git Branching](https://learngitbranching.js.org/) (interactive, beginner): the best way to understand branches and rebasing visually.
- [Software Carpentry: Version Control with Git](https://swcarpentry.github.io/git-novice/) (lesson, beginner): written for researchers.
- [Chris Beams, *How to Write a Git Commit Message*](https://cbea.ms/git-commit/) and [Conventional Commits](https://www.conventionalcommits.org/en/v1.0.0/) (articles, beginner).

---

[← 12 · Capstone](12_capstone.ipynb) &nbsp;|&nbsp; [Course overview](../README.md) &nbsp;|&nbsp; [X2 · Building a scikit-learn compatible estimator →](X2_sklearn_compatible_estimator.ipynb)